<a href="https://colab.research.google.com/github/Thrna06/Tranformers-/blob/main/AI_StudyMate_Transformer_LLM_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 AI StudyMate — Transformer + LLM + RAG

A Google Colab project using a Transformer embedding model, FLAN-T5 LLM, FAISS semantic search, PDF processing, RAG, summarization, and MCQ generation.


## Recommended Colab setting

Go to **Runtime → Change runtime type → T4 GPU** if available.

The notebook also works on CPU, but generation will be slower.

In [1]:
# Cell 1 — Install dependencies
!pip -q install -U transformers sentence-transformers faiss-cpu pypdf accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.2 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports and GPU check
import re
import torch
import faiss
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
GPU available: False


In [3]:
# Cell 3 — Load the Transformer embedding model and LLM

# Transformer used for semantic embeddings
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# Lightweight Transformer-based LLM; easier to run in Colab
LLM_MODEL = 'google/flan-t5-small'

print('Loading embedding Transformer...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print('Embedding Transformer loaded!')

print('Loading FLAN-T5 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

print('Loading FLAN-T5 model...')
model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)

if torch.cuda.is_available():
    device = torch.device('cuda')
    model = model.to(device)
    print('Using GPU:', torch.cuda.get_device_name(0))
else:
    device = torch.device('cpu')
    print('Using CPU')

print('\n✅ All models loaded successfully!')

Loading embedding Transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Using CPU

✅ All models loaded successfully!


In [4]:
# Cell 4 — Direct FLAN-T5 generation
# We intentionally do NOT use pipeline('text2text-generation'),
# because current Colab/Transformers versions may not expose that task.

def generate_text(prompt, max_new_tokens=150):
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompt = 'Explain artificial intelligence in simple words.'
print(generate_text(test_prompt, max_new_tokens=100))

Artificial intelligence is a tool used to measure the size of a machine.


In [5]:
# Cell 5 — Upload a PDF
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print('Uploaded:', pdf_file)

Saving pdf&rendition=1.pdf to pdf&rendition=1.pdf
Uploaded: pdf&rendition=1.pdf


In [6]:
# Cell 6 — Extract text from PDF

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ''
        if page_text.strip():
            pages.append(f'[Page {page_number}]\n{page_text}')

    return '\n\n'.join(pages)

document_text = extract_pdf_text(pdf_file)

print('Characters extracted:', len(document_text))
print('\nPreview:\n')
print(document_text[:2000])

Characters extracted: 68050

Preview:

[Page 1]
BCS502                            MODULE -01                                                            AZ Documents 
 
www.azdocuments.in                                                                                                  1 
 
 
MODULE-1 
COMPUTER NETWORKS  
Course Code BCS502 
 
Introduction: Data Communications, Networks, Network Types, Networks Models: 
Protocol Layering, TCP/IP Protocol suite, The OSI model, Introduction to Physical 
Layer: Transmission media, Guided Media, Unguided Media: Wireless. Switching: 
Packet Switching and its types. Textbook: Ch. 1.1 - 1.3, 2.1 - 2.3, 7.1 – 7.3, 8.3 
 
1.1 DATA COMMUNICATIONS  
Data communications are the exchange of data between two devices via some form of 
transmission medium such as a wire cable. For data communications to occur, the 
communicating devices must be part of a communication system made up of a combination of 
hardware (physical equipment) and software (program

In [7]:
# Cell 7 — Clean and chunk the document

def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip()

def create_chunks(text, chunk_size=700, overlap=100):
    text = clean_text(text)
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

chunks = create_chunks(document_text)
print('Number of chunks:', len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f'\n--- CHUNK {i} ---')
    print(chunk[:700])

Number of chunks: 87

--- CHUNK 1 ---
[Page 1] BCS502 MODULE -01 AZ Documents www.azdocuments.in 1 MODULE-1 COMPUTER NETWORKS Course Code BCS502 Introduction: Data Communications, Networks, Network Types, Networks Models: Protocol Layering, TCP/IP Protocol suite, The OSI model, Introduction to Physical Layer: Transmission media, Guided Media, Unguided Media: Wireless. Switching: Packet Switching and its types. Textbook: Ch. 1.1 - 1.3, 2.1 - 2.3, 7.1 – 7.3, 8.3 1.1 DATA COMMUNICATIONS Data communications are the exchange of data between two devices via some form of transmission medium such as a wire cable. For data communications to occur, the communicating devices must be part of a communication system made up of a combination o

--- CHUNK 2 ---
o occur, the communicating devices must be part of a communication system made up of a combination of hardware (physical equipment) and software (programs). The effectiveness of a data communications system depends on four fundamental character

In [8]:
# Cell 8 — Create Transformer embeddings

print('Creating embeddings...')
embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

print('Embedding shape:', embeddings.shape)

Creating embeddings...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (87, 384)


In [9]:
# Cell 9 — Build FAISS vector index

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(embeddings.astype('float32'))

print('FAISS index created!')
print('Vectors stored:', index.ntotal)

FAISS index created!
Vectors stored: 87


In [10]:
# Cell 10 — Semantic retrieval

def search_documents(question, k=3):
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        question_embedding.astype('float32'),
        min(k, len(chunks))
    )

    return [chunks[i] for i in indices[0] if i < len(chunks)]

results = search_documents('What is the main topic of this document?')
for i, result in enumerate(results, start=1):
    print(f'\n--- RESULT {i} ---\n{result}')


--- RESULT 1 ---
Documents www.azdocuments.in 35 ○ Provides services to the user ○ Examples: file transfer, email, web access 2. Presentation Layer (Layer 6) ○ Data formatting ○ Encryption and decryption ○ Data compression 3. Session Layer (Layer 5) ○ Establishes, manages, and terminates sessions ○ Controls dialog between applications 4. Transport Layer (Layer 4) ○ End-to-end delivery ○ Error control and flow control 5. Network Layer (Layer 3) ○ Logical addressing ○ Routing and path selection 6. Data Link Layer (Layer 2) ○ Framing ○ Error detection ○ MAC addressing 7. Physical Layer (Layer 1) ○ Transmission of raw bits ○ Defines cables, signals, voltages [Page 36] BCS502 MODULE -01 AZ Documents www.azdocume

--- RESULT 2 ---
ality in the video is the result. 1.1.1 Components A data communications system has five components (see Figure 1.1). [Page 2] BCS502 MODULE -01 AZ Documents www.azdocuments.in 2 1. Message. The message is the information (data) to be communicated. Popular forms o

In [11]:
# Cell 11 — RAG question answering

def generate_answer(question):
    relevant_chunks = search_documents(question, k=3)
    context = '\n\n'.join(relevant_chunks)

    prompt = f'''Answer the question using only the context below.
If the answer is not present in the context, say you could not find it in the uploaded document.

Context:
{context}

Question:
{question}

Answer:'''

    return generate_text(prompt, max_new_tokens=180)

question = input('Ask a question about your PDF: ')
print('\nAI ANSWER:\n')
print(generate_answer(question))

Ask a question about your PDF: what is CN

AI ANSWER:

Layer-to-layer) communication


In [ ]:
# Cell 12 — Generate a summary

def generate_summary():
    # Limit context for the lightweight Colab model
    context = '\n\n'.join(chunks[:6])

    prompt = f'''Summarize the following study material.
Include the main topic, important concepts, definitions, and key points.

Study material:
{context}

Summary:'''

    return generate_text(prompt, max_new_tokens=300)

print(generate_summary())

In [12]:
# Cell 13 — Generate MCQs

def generate_quiz():
    context = '\n\n'.join(chunks[:5])

    prompt = f'''Create 5 multiple-choice questions from the study material.
For each question give A, B, C, D and the correct answer.

Study material:
{context}

Quiz:'''

    return generate_text(prompt, max_new_tokens=500)

print(generate_quiz())

How many times is the delay in the delivery of audio or video packets?


In [18]:
# Cell 14 — Simple Gradio interface
import gradio as gr

def ask_question(question):
    if not question.strip():
        return 'Please enter a question.'
    return generate_answer(question)

with gr.Blocks(title='AI StudyMate') as app:
    gr.Markdown('# 🤖 AI StudyMate\n### Transformer + LLM + RAG')

    question = gr.Textbox(
        label='Ask a question about your PDF',
        placeholder='Example: Explain this topic in simple words.'
    )
    ask_button = gr.Button('Ask AI')
    answer = gr.Textbox(label='AI Answer', lines=8)
    ask_button.click(ask_question, inputs=question, outputs=answer)

    gr.Markdown('## 📚 Study Tools')
    summary_button = gr.Button('Generate Summary')
    summary_output = gr.Textbox(label='Summary', lines=10)
    summary_button.click(generate_summary, outputs=summary_output)

    quiz_button = gr.Button('Generate MCQ Quiz')
    quiz_output = gr.Textbox(label='Quiz', lines=15)
    quiz_button.click(generate_quiz, outputs=quiz_output)

app.launch(share=True)

NameError: name 'generate_summary' is not defined